In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/spam_preprocessed.csv')

# Drop any rows where cleaning produced empty text
df = df.dropna(subset=['cleaned_message'])
df = df[df['cleaned_message'].str.strip() != '']

print(df.shape)
df[['cleaned_message', 'label_num']].head()

(5150, 5)


,cleaned_message,label_num
0,jurong point crazi avail bugi great world buff...,0
1,lar joke wif oni,0
2,free entri wkli comp win cup final tkt 21st ma...,1
3,dun say earli hor alreadi say,0
4,nah dont think goe usf live around though,0


In [2]:
X = df['cleaned_message']   # input (the text)
y = df['label_num']         # target (0=ham, 1=spam)

print("X shape:", X.shape)
print("y distribution:\n", y.value_counts())

X shape: (5150,)
y distribution:
 label_num
0    4497
1     653
Name: count, dtype: int64


TRAIN/TEST SPILIT - BEFORE VECTORIZE

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% held out for testing
    random_state=42,      # reproducibility
    stratify=y            # preserve class balance in both splits
)

print("Train size:", X_train.shape[0])
print("Test size: ", X_test.shape[0])
print("\nTrain spam %:", y_train.mean() * 100)
print("Test spam %: ", y_test.mean() * 100)

Train size: 4120
Test size:  1030

Train spam %: 12.669902912621358
Test spam %:  12.718446601941746


 fit TF-IDF — on TRAINING data only

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create the vectorizer
tfidf = TfidfVectorizer(max_features=3000)

# FIT on training data, then TRANSFORM it
X_train_tfidf = tfidf.fit_transform(X_train)

# Only TRANSFORM the test data (do NOT fit!)
X_test_tfidf = tfidf.transform(X_test)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Test TF-IDF shape: ", X_test_tfidf.shape)

Train TF-IDF shape: (4120, 3000)
Test TF-IDF shape:  (1030, 3000)


In [5]:
# What does the vocabulary look like?
print("Vocabulary size:", len(tfidf.vocabulary_))
print("\nSample features:", tfidf.get_feature_names_out()[:20])

# The output is a sparse matrix (mostly zeros)
print("\nMatrix type:", type(X_train_tfidf))
print("Density:", X_train_tfidf.nnz / (X_train_tfidf.shape[0] * X_train_tfidf.shape[1]) * 100, "% non-zero")

Vocabulary size: 3000

Sample features: ['008704050406' '0089mi' '0121' '01223585236' '01223585334' '0125698789'
 '020603' '0207' '02070836089' '02072069400' '02073162414' '020903' '021'
 '050703' '0578' '060505' '061104' '071104' '0776xxxxxxx' '07xxxxxxxxx']

Matrix type: <class 'scipy.sparse._csr.csr_matrix'>
Density: 0.2304611650485437 % non-zero


In [6]:
# Take the first training message and show its top weighted words
import numpy as np

feature_names = tfidf.get_feature_names_out()
first_msg_vector = X_train_tfidf[0].toarray().flatten()

# Get top 10 highest-weighted words in this message
top_indices = first_msg_vector.argsort()[-10:][::-1]

print("Original:", X_train.iloc[0])
print("\nTop weighted words:")
for idx in top_indices:
    if first_msg_vector[idx] > 0:
        print(f"  {feature_names[idx]}: {first_msg_vector[idx]:.3f}")

Original: gam gone outstand inning

Top weighted words:
  gam: 0.538
  outstand: 0.512
  inning: 0.512
  gone: 0.431


Next Phase 

Train Multinomial navie bayes

In [7]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score

# Create and train
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)

# Predict on the test set
nb_preds = nb.predict(X_test_tfidf)

# Quick scores
print("Naive Bayes")
print("Accuracy:", accuracy_score(y_test, nb_preds))
print("F1 score:", f1_score(y_test, nb_preds))

Naive Bayes
Accuracy: 0.9718446601941747
F1 score: 0.8755364806866953


Train Logistic Regression 

In [8]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_tfidf, y_train)

lr_preds = lr.predict(X_test_tfidf)

print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, lr_preds))
print("F1 score:", f1_score(y_test, lr_preds))

Logistic Regression
Accuracy: 0.958252427184466
F1 score: 0.8088888888888889


Train Linear SVM

In [9]:
from sklearn.svm import LinearSVC

svm = LinearSVC()
svm.fit(X_train_tfidf, y_train)

svm_preds = svm.predict(X_test_tfidf)

print("Linear SVM")
print("Accuracy:", accuracy_score(y_test, svm_preds))
print("F1 score:", f1_score(y_test, svm_preds))

Linear SVM
Accuracy: 0.9815533980582525
F1 score: 0.9236947791164659


Comparsion Table

In [10]:
from sklearn.metrics import precision_score, recall_score

models = {
    'Naive Bayes': nb_preds,
    'Logistic Regression': lr_preds,
    'Linear SVM': svm_preds
}

results = []
for name, preds in models.items():
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, preds),
        'Precision': precision_score(y_test, preds),
        'Recall': recall_score(y_test, preds),
        'F1': f1_score(y_test, preds)
    })

results_df = pd.DataFrame(results).round(4)
results_df

,Model,Accuracy,Precision,Recall,F1
0,Naive Bayes,0.9718,1.0000,0.7786,0.8755
1,Logistic Regression,0.9583,0.9681,0.6947,0.8089
2,Linear SVM,0.9816,0.9746,0.8779,0.9237


Cross-validation

In [12]:
from sklearn.model_selection import cross_val_score

# Re-vectorize the full dataset for CV (fit happens inside each fold automatically when using a pipeline,
# but for a quick check we'll vectorize X here)
X_full_tfidf = tfidf.transform(X)  # using the vocabulary already fit on train

for name, model in [('Naive Bayes', MultinomialNB()),
                    ('Logistic Regression', LogisticRegression(max_iter=1000)),
                    ('Linear SVM', LinearSVC())]:
    scores = cross_val_score(model, X_full_tfidf, y, cv=5, scoring='f1')
    print(f"{name}: F1 = {scores.mean():.4f} (+/- {scores.std():.4f})")

Naive Bayes: F1 = 0.8718 (+/- 0.0224)
Logistic Regression: F1 = 0.7721 (+/- 0.0470)
Linear SVM: F1 = 0.9013 (+/- 0.0192)


Build a Pipeline 

In [13]:
from sklearn.pipeline import Pipeline

# Pick the best model from your comparison (likely SVM or LogReg)
final_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=3000)),
    ('classifier', LinearSVC())
])

# Train on the RAW training text (pipeline handles vectorization internally)
final_pipeline.fit(X_train, y_train)

# Evaluate
pipeline_preds = final_pipeline.predict(X_test)
print("Pipeline F1:", f1_score(y_test, pipeline_preds))

Pipeline F1: 0.9236947791164659
